# 03 — Graphs, shortest paths and network skimming

Every path-based computation in AequilibraE runs on a **Graph** — a compiled,
Cython-backed representation of the network for one mode. In this notebook we:

1. build graphs from the Sioux Falls project;
2. compute a shortest path between two nodes and map it;
3. **skim** the network: compute zone-to-zone cost matrices (time, distance);
4. store the skims in the project.

Skim matrices are the backbone of demand modeling: trip distribution (notebook 04)
consumes them as impedance.


In [1]:
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

from aequilibrae.utils.create_example import create_example

fldr = str(Path(gettempdir()) / uuid4().hex)
project = create_example(fldr, "sioux_falls")

project.network.build_graphs()
graph = project.network.graphs["c"]        # 'c' = car

# Minimise free-flow time; skim both time and distance along the way
graph.set_graph("free_flow_time")
graph.set_skimming(["free_flow_time", "distance"])

# Sioux Falls quirk: all nodes are centroids, so allow paths through centroids
graph.set_blocked_centroid_flows(False)
graph

C:\Users\Riz\Desktop\AequilibraE\aequilibrae\aequilibrae\paths\graph.py:218: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  build_compressed_graph(self, remove_dead_ends)
C:\Users\Riz\Desktop\AequilibraE\aequilibrae\aequilibrae\paths\graph.py:218: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are settin

## A single shortest path

`PathResults` computes one origin's tree and lets us extract the path to any destination.


In [2]:
from aequilibrae.paths import PathResults

res = PathResults()
res.prepare(graph)
res.compute_path(1, 17)     # from node 1 to node 17

print("nodes :", res.path_nodes)
print("links :", res.path)
print(f"cost  : {res.milepost[-1]:.2f} minutes")

nodes : [ 1  2  6  8 16 17]
links : [ 1  4 16 22 49]
cost  : 20.00 minutes


In [3]:
# JupyterGIS map helper ------------------------------------------------------
# GISDocument is JupyterGIS' notebook API: it builds a live, QGIS-like map
# document rendered directly in JupyterLab. Layers added from GeoDataFrames
# are converted to GeoJSON on the fly.
#
# add_gdf also translates the declarative symbology into the OpenLayers
# flat-style expressions the current JupyterGIS frontend renders from, so
# colours and line widths show up without touching the symbology panel.
import json

import matplotlib.colors
import matplotlib.pyplot as _plt
from jupytergis import GISDocument
from jupytergis_lab.notebook.symbology import to_symbology_state

OSM_TILES = "https://tile.openstreetmap.org/{z}/{x}/{y}.png"

def new_map(gdf_for_extent=None, zoom=12):
    """Create a GISDocument centred on a layer, with an OpenStreetMap basemap."""
    kwargs = {}
    if gdf_for_extent is not None:
        b = gdf_for_extent.total_bounds  # (minx, miny, maxx, maxy)
        kwargs = {"longitude": (b[0] + b[2]) / 2, "latitude": (b[1] + b[3]) / 2, "zoom": zoom}
    doc = GISDocument(**kwargs)
    doc.add_raster_layer(OSM_TILES, name="OpenStreetMap", attribution="(C) OpenStreetMap contributors", opacity=0.6)
    return doc

def _hex(rgba):
    return matplotlib.colors.to_hex(rgba)

def _ramp_expr(fld, params):
    name, dom = params.get("name", "viridis"), params.get("domain") or [0.0, 1.0]
    cmap = _plt.get_cmap(name)
    if params.get("reverse"):
        cmap = cmap.reversed()
    expr = ["interpolate", ["linear"], ["get", fld]]
    for i in range(7):
        t = i / 6
        expr += [dom[0] + t * (dom[1] - dom[0]), _hex(cmap(t))]
    return expr

def _scalar_expr(fld, params):
    d, r = params["domain"], params["range"]
    return ["interpolate", ["linear"], ["get", fld], d[0], r[0], d[1], r[1]]

def _cat_expr(fld, params, gdf):
    cmap = _plt.get_cmap(params.get("colorRamp", "tab10"))
    vals = list(dict.fromkeys(gdf[fld].dropna()))
    expr = ["match", ["get", fld]]
    for i, v in enumerate(vals):
        expr += [v, _hex(cmap(i % cmap.N))]
    return expr + ["#9ca3af"]

def _flat_style(symbology, gdf):
    """Grammar symbology -> OpenLayers flat-style dict (what the map renders)."""
    state = to_symbology_state(symbology)
    if not state:
        return None
    flat = {}
    for layer in state.get("layers", []):
        for rule in layer.get("rules", []):
            flds = rule.get("fields") or [None]
            for m in rule.get("mappings", []):
                scheme = m["scale"]["scheme"]
                params = m["scale"].get("params", {})
                if scheme == "constant_rgba":
                    val = params["value"]
                    val = _hex([c if c <= 1 else c / 255 for c in val]) if isinstance(val, (list, tuple)) else val
                elif scheme == "constant_num":
                    val = params["value"]
                elif scheme == "colorMap":
                    val = _ramp_expr(flds[0], params)
                elif scheme == "scalar":
                    val = _scalar_expr(flds[0], params)
                elif scheme == "categorical":
                    val = _cat_expr(flds[0], params, gdf)
                else:
                    continue
                for enc in m.get("encodings", []):
                    flat[enc] = val
    if any(k.startswith("circle") for k in flat) and "circle-radius" not in flat:
        flat["circle-radius"] = 5
    if "stroke-color" in flat and "stroke-width" not in flat:
        flat["stroke-width"] = 1.5
    return flat or None

def merge_lines(gdf, tol=0.01):
    """Collapse many lines into a single MultiLineString feature.

    Backdrop and class-display layers do not need per-feature identity, and one
    merged feature is a fraction of the size of tens of thousands of features -
    which keeps national-scale layers inside the notebook sync message limit.
    """
    import geopandas as _gpd
    from shapely.geometry import MultiLineString
    parts = []
    for geom in gdf.geometry.simplify(tol):
        if geom is None or geom.is_empty:
            continue
        parts.extend(geom.geoms if geom.geom_type == "MultiLineString" else [geom])
    return _gpd.GeoDataFrame({"links": [len(parts)]}, geometry=[MultiLineString(parts)], crs=gdf.crs)

def _round_coords(o, nd=5):
    if isinstance(o, (int, float)):
        return round(o, nd)
    if isinstance(o, list):
        return [_round_coords(v, nd) for v in o]
    return o

def add_gdf(doc, gdf, name, symbology=None, **kwargs):
    """Add a GeoDataFrame to the map as a GeoJSON layer, with rendered symbology.

    Coordinates are quantized to ~1 m so even national-scale layers stay well
    under Jupyter's websocket message limit (oversized layers are dropped
    silently by the sync, so this matters more than it looks).
    """
    data = json.loads(gdf.to_json())
    for f in data.get("features", []):
        g = f.get("geometry")
        if g and "coordinates" in g:
            g["coordinates"] = _round_coords(g["coordinates"])
    lid = doc.add_geojson_layer(data=data, name=name,
                                symbology=symbology, **kwargs)
    flat = _flat_style(symbology, gdf)
    if flat:
        layer = doc._layers.get(lid)
        layer["parameters"]["color"] = flat
        doc._layers[lid] = layer
    return lid

In [4]:
from jupytergis_lab.notebook.symbology import constant

links = project.network.links.data
path_links = links[links.link_id.isin(res.path)]

doc = new_map(links, zoom=12)
add_gdf(doc, links, "network", opacity=0.5, symbology=[[constant("#94a3b8").encoding("stroke")]])
add_gdf(doc, path_links, "shortest path", symbology=[[constant("#dc2626").encoding("stroke")]])
doc

C:\Users\Riz\AppData\Local\Temp\ipykernel_18188\1581409630.py:24: UserWarning: The JupyterGIS Python API is better experienced in the xeus-python kernel which supports awaiting comm messages
  doc = GISDocument(**kwargs)


## Skimming the whole network

`NetworkSkimming` runs one shortest-path tree per origin (in parallel) and collects
the skimmed fields into a zone-by-zone matrix.


In [5]:
from aequilibrae.paths import NetworkSkimming

skm = NetworkSkimming(graph)
skm.execute()

skims = skm.results.skims
print(skims.names)          # one matrix core per skimmed field
skims.get_matrix("free_flow_time")[:5, :5]

                                                  :   0%|          | 0/24 [00:00<?, ?it/s]

['free_flow_time', 'distance']


array([[ 0.,  6.,  4.,  8., 10.],
       [ 6.,  0., 10., 11.,  9.],
       [ 4., 10.,  0.,  4.,  6.],
       [ 8., 11.,  4.,  0.,  2.],
       [10.,  9.,  6.,  2.,  0.]])

In [6]:
# Persist into the project so later notebooks (and colleagues) can reuse them
skm.save_to_project("base_skims")
project.matrices.list()[["name", "file_name", "cores"]]

,name,file_name,cores
0,demand_omx,demand.omx,1
1,demand_mc,demand_mc.omx,3
2,skims,skims.omx,2
3,demand_aem,demand.aem,1
4,base_skims,base_skims.omx,2


In [7]:
project.close()

---
**Next:** [04 — Trip distribution](04_trip_distribution.ipynb): turning trip totals into
a full origin-destination matrix with gravity models and IPF.
